In [ ]:
!pip install pandas
!pip install numpy
!pip install tensorflow
!pip install scikit-learn
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 38.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 85.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [ ]:
import pandas as pd

In [ ]:
data = pd.read_json("News_Category_Dataset_v3.json", lines = True)

In [ ]:
data = data['short_description'].head(50000)

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [ ]:
tokenizer = Tokenizer(oov_token='<OOV>')

In [ ]:
tokenizer.fit_on_texts(data)

In [ ]:
data = tokenizer.texts_to_sequences(data)

In [ ]:
input_sequences = []

for seq in data:
    if len(seq) < 2:
        continue
    for i in range(1, len(seq)):
        ngram = seq[:i+1]
        input_sequences.append(ngram)

In [ ]:
max_len = max([len(i) for i in input_sequences])

In [ ]:
input = pad_sequences(input_sequences, padding='pre', maxlen=max_len)

In [ ]:
X = input[:,:-1]
Y = input[:,-1]

In [ ]:
Y = np.array(Y)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
vocab_size = len(tokenizer.word_index)+1

In [ ]:
max_len = X.shape[1]

In [ ]:
model = Sequential([Embedding(vocab_size, 128, input_length=max_len), LSTM(256), Dense(vocab_size, activation = 'softmax')])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.build(input_shape=(None, max_len))

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 77, 128)        │     5,124,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 40039)          │    10,290,023 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,809,255 (60.31 MB)

 Trainable params: 15,809,255 (60.31 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
model.fit(X_train, Y_train, epochs=30, validation_data=(X_test, Y_test))

Epoch 1/30
17216/17216 ━━━━━━━━━━━━━━━━━━━━ 433s 25ms/step - accuracy: 0.1090 - loss: 6.9889 - val_accuracy: 0.1363 - val_loss: 6.5711
Epoch 2/30
17216/17216 ━━━━━━━━━━━━━━━━━━━━ 429s 25ms/step - accuracy: 0.1532 - loss: 6.2138 - val_accuracy: 0.1522 - val_loss: 6.4093
Epoch 3/30
17216/17216 ━━━━━━━━━━━━━━━━━━━━ 440s 25ms/step - accuracy: 0.1762 - loss: 5.7828 - val_accuracy: 0.1594 - val_loss: 6.4324
Epoch 4/30
17216/17216 ━━━━━━━━━━━━━━━━━━━━ 431s 25ms/step - accuracy: 0.1970 - loss: 5.3810 - val_accuracy: 0.1597 - val_loss: 6.5276
Epoch 5/30
17216/17216 ━━━━━━━━━━━━━━━━━━━━ 466s 27ms/step - accuracy: 0.2184 - loss: 5.0098 - val_accuracy: 0.1589 - val_loss: 6.6611
Epoch 6/30
17216/17216 ━━━━━━━━━━━━━━━━━━━━ 429s 25ms/step - accuracy: 0.2411 - loss: 4.6682 - val_accuracy: 0.1533 - val_loss: 6.8709
Epoch 7/30
17216/17216 ━━━━━━━━━━━━━━━━━━━━ 427s 25ms/step - accuracy: 0.2658 - loss: 4.3739 - val_accuracy: 0.1515 - val_loss: 7.0294
Epoch 8/30
17216/17216 ━━━━━━━━━━━━━━━━━━━━ 427s 25ms/s

KeyboardInterrupt: 

In [24]:
model.save('nextword_model.h5')

In [25]:
import pickle

In [26]:
with open('tokenizer.pkl', 'wb') as file:
    pickle.dump(tokenizer, file)